# 02 — 結果集計 & 学習曲線プロット

Driveに蓄積した `results/*.json` を読み込み、訓練サイズ vs MRE/SDR をプロットする。
全ジョブが揃っていなくても途中経過として実行できる。

In [ ]:
DRIVE_LC_DIR = "/content/drive/MyDrive/anglist_learning_curve"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

results_dir = os.path.join(DRIVE_LC_DIR, "results")

# バリアント別に読み込む（サブディレクトリ構造）
all_results = []
variants_found = []
for entry in sorted(os.listdir(results_dir)):
    sub = os.path.join(results_dir, entry)
    if os.path.isdir(sub):
        variants_found.append(entry)
        for fname in sorted(os.listdir(sub)):
            if fname.endswith(".json"):
                with open(os.path.join(sub, fname)) as f:
                    all_results.append(json.load(f))

print(f"バリアント: {variants_found}")
print(f"読み込み済みジョブ数: {len(all_results)}")


In [ ]:
# バリアント × サイズ別に集計（mean ± std over folds）
ANGLE_NAMES = ["PI", "PT", "SS", "LL"]

# variant -> size -> {mre, angle_mae per angle, mre_excl}
var_size_data = defaultdict(lambda: defaultdict(lambda: {
    "mre": [], "mre_excl": [], "angle_mae": {a: [] for a in ANGLE_NAMES},
    "angle_mae_excl": {a: [] for a in ANGLE_NAMES}
}))

for r in all_results:
    variant = r.get("variant", "unknown")
    s = r["size"]
    d = var_size_data[variant][s]
    d["mre"].append(r["overall"]["mre_mm"])
    if "angle_mae" in r["overall"]:
        for a in ANGLE_NAMES:
            if a in r["overall"]["angle_mae"]:
                d["angle_mae"][a].append(r["overall"]["angle_mae"][a])
    if "overall_excl_outliers" in r and r["overall_excl_outliers"]["mre_mm"] is not None:
        d["mre_excl"].append(r["overall_excl_outliers"]["mre_mm"])
        if "angle_mae" in r["overall_excl_outliers"]:
            for a in ANGLE_NAMES:
                if a in r["overall_excl_outliers"]["angle_mae"]:
                    d["angle_mae_excl"][a].append(r["overall_excl_outliers"]["angle_mae"][a])

# 出力テーブル
for variant in sorted(var_size_data.keys()):
    print(f"\n=== {variant} ===")
    print(f"{'Size':>6}  {'MRE':>9}  {'MRE_excl':>10}  {'PI':>7}  {'PT':>7}  {'SS':>7}  {'LL':>7}  {'N':>3}")
    sizes = sorted(var_size_data[variant].keys())
    for s in sizes:
        d = var_size_data[variant][s]
        mre = f"{np.mean(d['mre']):.1f}" if d["mre"] else "-"
        mre_e = f"{np.mean(d['mre_excl']):.1f}" if d["mre_excl"] else "-"
        angle_strs = []
        for a in ANGLE_NAMES:
            v = d["angle_mae"][a]
            angle_strs.append(f"{np.mean(v):.1f}" if v else "-")
        n = len(d["mre"])
        print(f"{s:>6}  {mre:>9}  {mre_e:>10}  {'  '.join(angle_strs)}  {n:>3}")


In [ ]:
# 人力誤差（inter_annotator_error.py から）
HUMAN_MRE = 2.52  # mm overall
HUMAN_ANGLES = {"PI": 3.09, "PT": 1.03, "SS": 2.80, "LL": 3.57}  # degrees MAE

COLORS = ["steelblue", "darkorange", "seagreen", "crimson", "purple", "gray"]
ANGLE_COLORS = {"PI": "steelblue", "PT": "darkorange", "SS": "seagreen", "LL": "crimson"}

variants = sorted(var_size_data.keys())
n_variants = len(variants)

# --- Plot 1: MRE 学習曲線（バリアント比較）---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Learning Curve (5-fold CV)", fontsize=14)

for ax_idx, (use_excl, ax_title) in enumerate([
    (False, "MRE vs Training Size"),
    (True,  "MRE vs Training Size (outliers excluded)")
]):
    ax = axes[ax_idx]
    for vi, variant in enumerate(variants):
        sizes = sorted(var_size_data[variant].keys())
        key = "mre_excl" if use_excl else "mre"
        means = [np.mean(var_size_data[variant][s][key]) for s in sizes if var_size_data[variant][s][key]]
        stds  = [np.std( var_size_data[variant][s][key]) for s in sizes if var_size_data[variant][s][key]]
        sz    = [s for s in sizes if var_size_data[variant][s][key]]
        if not sz: continue
        ax.plot(sz, means, "o-", color=COLORS[vi], label=variant)
        ax.fill_between(sz, [m-s for m,s in zip(means,stds)], [m+s for m,s in zip(means,stds)],
                        alpha=0.15, color=COLORS[vi])
    ax.axhline(HUMAN_MRE, color="red", linestyle="--", linewidth=1.5, label=f"Human inter-rater ({HUMAN_MRE:.1f}mm)")
    ax.set_xlabel("Training samples"); ax.set_ylabel("MRE (mm)")
    ax.set_title(ax_title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_LC_DIR, "learning_curve_mre.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- Plot 2: 角度MAE学習曲線 ---
has_angle_data = any(
    any(var_size_data[v][s]["angle_mae"]["PI"] for s in var_size_data[v])
    for v in variants
)
if has_angle_data:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Angle MAE vs Training Size (5-fold CV)", fontsize=14)
    for ai, angle in enumerate(["PI", "PT", "SS", "LL"]):
        ax = axes[ai // 2][ai % 2]
        for vi, variant in enumerate(variants):
            sizes = sorted(var_size_data[variant].keys())
            vals = [var_size_data[variant][s]["angle_mae"][angle] for s in sizes]
            sz   = [s for s, v in zip(sizes, vals) if v]
            means = [np.mean(v) for v in vals if v]
            stds  = [np.std(v)  for v in vals if v]
            if not sz: continue
            ax.plot(sz, means, "o-", color=COLORS[vi], label=variant)
            ax.fill_between(sz, [m-s for m,s in zip(means,stds)], [m+s for m,s in zip(means,stds)],
                            alpha=0.15, color=COLORS[vi])
        ax.axhline(HUMAN_ANGLES[angle], color="red", linestyle="--", linewidth=1.5,
                   label=f"Human inter-rater ({HUMAN_ANGLES[angle]:.1f}°)")
        ax.set_xlabel("Training samples"); ax.set_ylabel("Angle MAE (°)")
        ax.set_title(f"{angle} MAE"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_LC_DIR, "learning_curve_angle_mae.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("角度MAEプロット保存完了")
else:
    print("⚠ 角度MAEデータなし（ノートブック更新後に再実行が必要）")


In [ ]:
# ランドマーク別MRE（バリアント別）
LANDMARK_ORDER = ["L1_ant", "L1_post", "S1_ant", "S1_post", "FH"]
HUMAN_LM = {"L1_ant": 1.97, "L1_post": 2.20, "S1_ant": 2.35, "S1_post": 1.84, "FH": 4.24}

fig, axes = plt.subplots(1, n_variants, figsize=(7 * n_variants, 5), squeeze=False)
fig.suptitle("Per-landmark MRE vs Training Size", fontsize=14)

lm_colors = ["steelblue", "darkorange", "seagreen", "crimson", "purple"]

for vi, variant in enumerate(variants):
    ax = axes[0][vi]
    lm_size_mre = defaultdict(lambda: defaultdict(list))
    for r in all_results:
        if r.get("variant", "unknown") != variant: continue
        for lm in LANDMARK_ORDER:
            if lm in r["landmarks"]:
                lm_size_mre[lm][r["size"]].append(r["landmarks"][lm]["mre_mm"])
    all_sizes = sorted({s for lm in lm_size_mre for s in lm_size_mre[lm]})
    for lm, color in zip(LANDMARK_ORDER, lm_colors):
        lm_sizes = sorted(lm_size_mre[lm].keys())
        lm_mean  = [np.mean(lm_size_mre[lm][s]) for s in lm_sizes]
        ax.plot(lm_sizes, lm_mean, "o-", label=lm, color=color)
        ax.axhline(HUMAN_LM[lm], linestyle=":", color=color, linewidth=1, alpha=0.6)
    ax.set_xlabel("Training samples"); ax.set_ylabel("MRE (mm)")
    ax.set_title(variant); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    if all_sizes: ax.set_xticks(all_sizes)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_LC_DIR, "learning_curve_per_landmark.png"), dpi=150, bbox_inches="tight")
plt.show()

# --- Outlier distribution ---
has_per_case = any("per_case_mre_mm" in r for r in all_results)
if has_per_case:
    all_case_mres = [m for r in all_results for m in r.get("per_case_mre_mm", [])]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(all_case_mres, bins=50, color="steelblue", alpha=0.7)
    import statistics
    med = statistics.median(all_case_mres)
    ax.axvline(med, color="red", linestyle="--", label=f"Median={med:.1f}mm")
    ax.axvline(med * 3, color="orange", linestyle="--", label=f"Outlier threshold (3×median)={med*3:.1f}mm")
    ax.set_xlabel("Per-case MRE (mm)"); ax.set_ylabel("Count")
    ax.set_title("Per-case MRE Distribution (all variants & folds)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_LC_DIR, "per_case_mre_histogram.png"), dpi=150, bbox_inches="tight")
    plt.show()
    n_outliers_total = sum(1 for m in all_case_mres if m > med * 3)
    print(f"全体外れ値数: {n_outliers_total}/{len(all_case_mres)} ({100*n_outliers_total/len(all_case_mres):.1f}%)")
